# Calculate mean benefit and change in admissions if stroke unit gains MT

Benefit is the difference in proportion of patients who have mRS 0-2 after treatment compared with no treatment. 

For each new MT stroke unit, calculate:
+ the admissions-weighted net benefit over the base scenario for all patients.
+ the total number of patients who benefit more with the new unit than in the base scenario, and
+ the mean benefit for these patients.

__Results__:

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

from classes.geography_processing import Geoprocessing
from classes.model import Model
from classes.scenario import Scenario
from classes.add_net_benefit import add_net_benefit

In [2]:
dir_data = 'data'
dir_output = 'output'
file_hospitals = 'stroke_hospitals_2025.csv'
file_scen_params = 'scenario_parameters.csv'
file_base_geo = 'results_geo_outcomes_base.csv'
file_base_results = 'results_full_base.csv'
file_benefit_by_team = 'benefit_of_each_team.csv'

## Load data

### Hospital info

In [3]:
p = os.path.join(dir_data, file_hospitals)
hospitals = pd.read_csv(p)

In [4]:
display(hospitals.head(3))

,SSNAP name,Postcode,Country,SSNAP Region,Easting,Northing,long,lat,Hospital_City,LSOA11CD,Use_IVT,Use_MT
0,John Radcliffe Hospital,OX39DU,England,"Buckinghamshire, Oxford and Berkshire",453940,207531,-1.219806,51.763871,Oxford,E01028534,1,1
1,Royal Berkshire Hospital,RG15AN,England,"Buckinghamshire, Oxford and Berkshire",472411,172962,-0.959330,51.451022,Reading,E01016414,1,0
2,Wycombe General Hospital,HP112TT,England,"Buckinghamshire, Oxford and Berkshire",486386,192694,-0.753428,51.626442,High Wycombe,E01017839,1,0


### Scenario parameters

In [5]:
p = os.path.join(dir_output, file_scen_params)

# Use converters to turn units column from strings to lists.
# Assumes that no postcodes contain spaces!
converters = {
    'new_mt_units': lambda x: (
        x.replace("'", '').replace(',', '')[1:-1].split()),
}
df_scen = pd.read_csv(p, converters=converters)

In [6]:
display(df_scen.head(6))

,scenario_name,scenario_geoprocess_name,new_mt_units,single_unit_changed,mt_travel_bias_minutes
0,base,base,[],0,0.1
1,base_nobias,base_nobias,[],0,0.0
2,taunworc,taunworc,"[TA15DA, WR51DD]",0,0.1
3,taunworc_nobias,taunworc_nobias,"[TA15DA, WR51DD]",0,0.0
4,only_RG15AN,only_RG15AN,[RG15AN],1,0.1
5,only_RG15AN_nobias,only_RG15AN_nobias,[RG15AN],1,0.0


Split into scenarios to calculate in full and scenarios to only calculate as summary:

In [7]:
m = df_scen['single_unit_changed'] == 0
df_scen_full = df_scen.loc[m].drop('single_unit_changed', axis='columns')
df_scen_sing = df_scen.loc[~m].drop('single_unit_changed', axis='columns')

In [8]:
display(df_scen_sing.head(5))

,scenario_name,scenario_geoprocess_name,new_mt_units,mt_travel_bias_minutes
4,only_RG15AN,only_RG15AN,[RG15AN],0.1
5,only_RG15AN_nobias,only_RG15AN_nobias,[RG15AN],0.0
6,only_HP112TT,only_HP112TT,[HP112TT],0.1
7,only_HP112TT_nobias,only_HP112TT_nobias,[HP112TT],0.0
8,only_CH21UL,only_CH21UL,[CH21UL],0.1


In [9]:
options_prop_lvo_dripship_treated_with_mt = [1.0, 0.6]
prop_lvo = 0.3

### Base scenario results

First load in base scenario results for a comparison:

In [10]:
p = os.path.join(dir_output, file_base_geo)
p2 = os.path.join(dir_output, file_base_results)
base_result = pd.concat(
    (pd.read_csv(p, index_col='LSOA'), pd.read_csv(p2, index_col='LSOA')),
    axis='columns'
)

Define names of columns with useful data:

In [11]:
col_admissions = 'Admissions_2025_Est'
# Get benefit for LVO
dict_cols_benefit = {}
for p in options_prop_lvo_dripship_treated_with_mt:
    s = '' if p == 1.0 else f'_transferloss{p}'.replace('.', '!')
    c = f'lvo_drip_ship_ivt_mt_mrs_0-2{s}'
    dict_cols_benefit[p] = c

# Rename base columns:
dict_cols_benefit_base = dict(
    [(c, f'{c}_base') for c in list(dict_cols_benefit.values())])
base_result = base_result.rename(columns=dict_cols_benefit_base)

In [12]:
print(dict_cols_benefit)

{1.0: 'lvo_drip_ship_ivt_mt_mrs_0-2', 0.6: 'lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6'}


Check contents:

In [13]:
display(base_result[
    [col_admissions] + list(dict_cols_benefit_base.values())
    ].head(5))

,Admissions_2025_Est,lvo_drip_ship_ivt_mt_mrs_0-2_base,lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base
LSOA,,,
ADUR001A,1.49403,0.429,0.429
ADUR001B,1.86762,0.429,0.429
ADUR001C,2.24110,0.376,0.356
ADUR001D,2.24110,0.429,0.429
ADUR001E,4.10862,0.377,0.356


Quick check of mean benefit in the base scenario:

In [14]:
print("Mean benefit weighted by admissions:")
for col, col_base in dict_cols_benefit_base.items():
    # Print mean weighted by admissions
    b = np.average(base_result[col_base], weights=base_result[col_admissions])
    print(f"{col_base:50s} {b:.3f}")

Mean benefit weighted by admissions:
lvo_drip_ship_ivt_mt_mrs_0-2_base                  0.389
lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6_base  0.377


## Main calculations

In [15]:
def calculate_geo_and_outcomes(
        mt_travel_bias,
        new_mt_units,
    ):
    # Process geographic data (only needed when hospital data changes):
    geo = Geoprocessing(limit_to_england=True,
                        mt_travel_bias=mt_travel_bias,
                        new_mt_centres=new_mt_units)
    geo.run()
    geodata = geo.combined_data.copy().reset_index()
    
    # Set up scenario
    scenario = Scenario({
        'name': 'temporary_name',
        'limit_to_england' : True
    })
    
    # Set up model
    model = Model(
        scenario=scenario,
        geodata=geodata
    )
    
    # Run model
    model.run()
    results = model.full_results
    return results

## Run for single-unit changes:

Don't save the full results to file. Gather summary results and save those.


In [16]:
hospital_net_benefit = {}
hospital_admissions_benefiting = {}
hospital_benefit_for_those_benefitting = {}

for p in options_prop_lvo_dripship_treated_with_mt:
    hospital_net_benefit[p] = {}
    hospital_admissions_benefiting[p] = {}
    hospital_benefit_for_those_benefitting[p] = {}

# i = 0
for scen_geo_name, df in df_scen_sing.groupby('scenario_geoprocess_name'):
    # These parameters should be the same for all df in group:
    mt_travel_bias = df.iloc[0]['mt_travel_bias_minutes']
    new_mt_units = df.iloc[0]['new_mt_units']

    # Shared geography and outcome results:
    df_geo_outcomes = calculate_geo_and_outcomes(
        mt_travel_bias,
        new_mt_units,
    )
    cols_start = df_geo_outcomes.columns

    results = df_geo_outcomes.copy()
    list_of_dfs = [results]
    for p in options_prop_lvo_dripship_treated_with_mt:
        # Pick out info for this scenario:
        s_p = '' if p == 1.0 else f'_transfer{p}'.replace('.', '!')
        # scen_name = f'{scen_geo_name}{s_p}'
    
        # Don't overwrite shared results:
        results_here = df_geo_outcomes.copy()
        # Calculate outcome results that depend on transfers:
        # Add net benefit columns
        results_here = add_net_benefit(
            results_here,
            prop_lvo=prop_lvo,
            prop_lvo_ds_mt=p
        )
    
        old_cols = sum([df.columns.tolist() for df in list_of_dfs], [])
        new_cols = [c for c in results_here.columns if c not in old_cols]
        if p != 1.0:
            rename_dict = dict([(c, f'{c}{p}'.replace('.', '!'))
                for c in new_cols])
        else:
            rename_dict = dict([(c, c) for c in new_cols])
        list_of_dfs.append(results_here[new_cols].rename(columns=rename_dict))
    
    results = pd.concat(list_of_dfs, axis='columns')
    cols_new = [c for c in results.columns if c not in cols_start]

    # Bring in base results:
    results = pd.concat(
        (results,
         base_result[list(dict_cols_benefit_base.values())]),
        axis='columns'
    ).reset_index()
        
    # Only keep useful columns:
    cols_to_keep = ([col_admissions] +
                    list(dict_cols_benefit_base.keys()) +
                    list(dict_cols_benefit_base.values()))
    results = results[cols_to_keep]
    # Compare benefit with base:
    for p, col_here in dict_cols_benefit.items():
        col_base = dict_cols_benefit_base[col_here]

        benefit_over_base_by_lsoa = results[col_here] - results[col_base]
    
        total_admissions = results[col_admissions].sum()
        hospital_net_benefit[p][scen_geo_name] = (
            np.average(benefit_over_base_by_lsoa, weights=results[col_admissions])
            if total_admissions > 0 else np.nan
        )
    
        benefiting_lsoas = benefit_over_base_by_lsoa > 0
        benefiting_benefit = benefit_over_base_by_lsoa.loc[benefiting_lsoas]
        benefiting_admissions = results[col_admissions].loc[benefiting_lsoas]
    
        hospital_admissions_benefiting[p][scen_geo_name] = benefiting_admissions.sum()
    
        benefiting_weight = benefiting_admissions.sum()
        hospital_benefit_for_those_benefitting[p][scen_geo_name] = (
            np.average(benefiting_benefit, weights=benefiting_admissions)
            if benefiting_weight > 0 else np.nan
        )
    # i += 1
    # if i > 4:
    #     print(oi)

The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS distributions might be improperly labelled.
The input mRS 

Set up for final results dataframe:

In [17]:
results_dfs = []

for p, col_here in dict_cols_benefit.items():
    s = '' if p == 1.0 else f' ({p} transferred)'.replace('.', '!')
    df = pd.DataFrame({
        'Hospital': list(hospital_net_benefit[p].keys()),
        'Weighted Net Benefit': list(hospital_net_benefit[p].values()),
        'Admission Catchment Benefiting': list(
            hospital_admissions_benefiting[p].values()),
        'Mean Benefit for Those Benefitting': list(
            hospital_benefit_for_those_benefitting[p].values()
        )
    })
    df = df.round(5)
    df['Admission Catchment Benefiting'] = (
        df['Admission Catchment Benefiting'].round(1))
    # Extra calculation:
    best_benefit = df['Weighted Net Benefit'].max()
    df['Relative Benefit'] = (
        df['Weighted Net Benefit'] / best_benefit
        if pd.notna(best_benefit) and best_benefit != 0 else np.nan
    ).round(3)

    # Tidy names.
    # Convert names from "only_Postcode" to "Postcode":
    df['Hospital'] = (
        df['Hospital'].str.split('_', expand=True)[1])
    df = df.set_index('Hospital')
    # Convert column names to show lossiness:
    rename_dict = dict([(c, f'{c}{s}') for c in df.columns])
    df = df.rename(columns=rename_dict)

    # Gather results.
    results_dfs.append(df)

Gather results into dataframe:

In [18]:
results_df = pd.concat(results_dfs, axis='columns').reset_index()

results_df = results_df.sort_values(by='Weighted Net Benefit',
                                    ascending=False)

# Merge in unit info:
results_df = pd.merge(
    hospitals[['Postcode', 'SSNAP name', 'Hospital_City']],
    results_df,
    how='right',
    left_on='Postcode',
    right_on='Hospital',
).set_index('SSNAP name').drop('Postcode', axis='columns')

## Check results

In [19]:
display(results_df.head(10))

,Hospital_City,Hospital,Weighted Net Benefit,Admission Catchment Benefiting,Mean Benefit for Those Benefitting,Relative Benefit,Weighted Net Benefit (0!6 transferred),Admission Catchment Benefiting (0!6 transferred),Mean Benefit for Those Benefitting (0!6 transferred),Relative Benefit (0!6 transferred)
SSNAP name,,,,,,,,,,
Northwick Park Hospital HASU,Harrow,HA13UJ,0.00129,48974.9,0.00261,1.000,0.00164,50266.5,0.00320,1.000
Midland Metropolitan University Hospital (MMUH) - Acute Stroke Services,Smethwick,B662QT,0.00105,48846.0,0.00217,0.814,0.00151,50220.9,0.00295,0.921
Watford General Hospital,Watford,WD180HB,0.00100,48349.9,0.00209,0.775,0.00117,49688.2,0.00237,0.713
Stepping Hill Hospital,Stockport,SK27JE,0.00098,46643.3,0.00213,0.760,0.00139,48169.1,0.00286,0.848
University Hospitals Dorset Stroke Service,Bournemouth,BH77DW,0.00097,46856.1,0.00211,0.752,0.00122,48209.7,0.00255,0.744
Northampton General Hospital,Northampton,NN15BD,0.00091,47541.8,0.00196,0.705,0.00106,49150.1,0.00219,0.646
Frimley Park Hospital,Frimley Park,GU167UJ,0.00090,47154.2,0.00196,0.698,0.00103,48703.7,0.00216,0.628
Musgrove Park Hospital,Taunton,TA15DA,0.00089,46953.3,0.00196,0.690,0.00096,48547.4,0.00204,0.585
New Cross Hospital,Wolverhampton,WV100QP,0.00084,47453.2,0.00183,0.651,0.00109,48954.3,0.00227,0.665


## Save results

In [20]:
p = os.path.join(dir_output, file_benefit_by_team)
results_df.to_csv(p, index=True)